# Pilot 5 · Readable traces of rank-1 SFT and GRPO

Implemented: bootstrap, raw corpus and CPU task-reference workflow. No training, production capture or learned readouts here.

Use a fresh **CPU runtime**. For the completed corpus, run **1–2 → 10–12**. Read the reference preview, then complete **13**. Cells **3–9** are earlier stages; reuse their saved evidence. Do not use Run all.


Ticket 04: CPU 1–2 → 14. Fresh GPU 1–2 → 4 → 15–16. Ticket 03 reference review remains independent. Do not Run all.


## 1. Mount storage and load the pinned harness

CPU or GPU. The harness has a separate checkout; your saved experiment artifacts stay in Drive.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/SPAR/spar-pilot/artifacts')
EXPERIMENT_NAME = 'pilot5-readable-traces-v1'
HARNESS_COMMIT = 'd7d1d24a67c7fa812113ff99ec5e4112c3478780'
REPO_DIR = Path('/content/pilot5-harness') / HARNESS_COMMIT

if not (REPO_DIR / '.git').is_dir():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', 'https://github.com/mahadikprasad15/spar-pilot.git',
                    str(REPO_DIR)], check=True)
    subprocess.run(['git', 'checkout', '--detach', HARNESS_COMMIT], cwd=REPO_DIR, check=True)
actual_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'],
                                        cwd=REPO_DIR, text=True).strip()
assert actual_commit == HARNESS_COMMIT, 'Harness checkout differs from its pin.'
loaded_package = sys.modules.get('pilot_eval')
if loaded_package and not Path(loaded_package.__file__).resolve().is_relative_to(REPO_DIR):
    raise RuntimeError('New harness release: restart the CPU runtime, then run 1–2 again.')
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from pilot_eval.notebook_session import NotebookSession
runner = NotebookSession(ARTIFACT_ROOT, REPO_DIR, plan_name=EXPERIMENT_NAME)
print('Installing CPU audit/corpus libraries; model libraries are not required here.', flush=True)
runner.run([sys.executable, '-u', '-m', 'pip', 'install', '-e',
            str(REPO_DIR) + '[trace-audit,trace-corpus]'])


## 2. Restore saved state

Read-only. No model imports or earlier experiment stages run. “Audited config” records a previous source audit; capture will revalidate the source evidence.


In [ ]:
from pilot_eval.trace_session import TraceSession

session = TraceSession(ARTIFACT_ROOT, REPO_DIR, EXPERIMENT_NAME)
state = session.restore()
print('Smoke:', state['smoke']['state'])
print('Colab validation:', state['target_validation'])
for label, source in state['sources'].items():
    print(label, '|', source['state'], '|', source['path'])
print('Capture contract:', len(state['sites']), 'named sites; no capture launched.')
print('Raw corpus:', state['corpus']['state'])


## 3. Select and audit existing sources (CPU)

These are the existing prepared source configurations. Change a path only when selecting another reviewed source. Auditing checks their existing seals and source evidence; it does not load weights or generate answers.


In [ ]:
SOURCE_CONFIGS = {
    'sft': ARTIFACT_ROOT / 'plans/pilot3-inputs-v1/activation.prepared.json',
    'grpo': ARTIFACT_ROOT / 'plans/pilot4-grpo-fp32-v1-writes-v1/activation.prepared.json',
    'random': ARTIFACT_ROOT / 'plans/pilot4-grpo-fp32-v1-random-writes-v1/activation.prepared.json',
}
session.select_sources(SOURCE_CONFIGS)
source_audit = session.audit_sources()
for label, source in source_audit.items():
    print(label, '|', source['model'], '| checkpoints:', source['checkpoint_steps'])


## 4. Install and check pinned GPU dependencies

GPU smoke only. Removes conflicting optional vision/audio packages and imports the actual text-model classes in a fresh process. No model weights are downloaded. If Colab requests a restart, run 1–2 again before continuing.


In [ ]:
from pilot_eval.sft_backend import PINS

assert not session.active_children(), 'An experiment child is still running.'
runner.run([sys.executable, '-m', 'pip', 'install',
                *[f'{package}=={version}' for package, version in PINS.items()]], check=True)
runner.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
                'torchvision', 'torchaudio'], check=True)
runtime = session.readiness()
print(json.dumps(runtime, indent=2))


## 5. Write the minimal Colab smoke artifact

Records actual import-check duration and GPU allocation, writes a small sealed artifact to Drive, then reads it back. After this cell finishes, restart the runtime. The GPU is not needed for cell 6.


In [ ]:
smoke = session.write_smoke()
print(json.dumps(smoke, indent=2))
print('Saved smoke:', session.directory)
print('Next: restart runtime, run 1–2, then 6.')


## 6. Verify storage after reconnect (CPU)

Run only in the fresh kernel after 1–2. Verifies the saved smoke seal and records a reconnect receipt. This qualifies bootstrap imports/storage only; capture and lens qualification are later tickets.


In [ ]:
restored = session.verify_reconnect()
print(json.dumps(restored['smoke'], indent=2))
print('Real-Colab bootstrap validation:', restored['target_validation'])


## 7. Review the raw corpus selection (CPU)

Requires cell 3’s saved source audit. Preserve the original 150 FineWeb identities; select 50 disjoint eligible documents from the same pinned 2,000-document pool. The seed and review are saved before sampling. “New-50” does not establish that documents were previously unseen.

Set `reviewed` to `True` only after approving the procedure and filling both notes. Legacy text/token discrepancies stop preparation; no replacement is automatic.


In [ ]:
state = session.restore()
assert all(state['sources'].get(label, {}).get('state', '').startswith('audited-config')
           for label in ['sft', 'grpo', 'random']), 'Run cell 3 to audit all three sources first.'

CORPUS_NAME = EXPERIMENT_NAME + '-raw-corpus-v1'
CORPUS_SOURCE = Path(state['sources']['sft']['path'])
CORPUS_REVIEW = {
    'schema_version': 1,
    'procedure': 'same-pool-disjoint-random-v1',
    'seed': 42,
    'reviewed': False,
    'notes': '',
    'prior_exposure': '',
}
# Notes describe the reviewed procedure. Exposure records known history or explicitly says unknown.
assert CORPUS_REVIEW['reviewed'] and CORPUS_REVIEW['notes'].strip() and CORPUS_REVIEW['prior_exposure'].strip(), (
    'Review the procedure and fill its notes/exposure history before preparation.'
)
CORPUS_REVIEW_PATH = ARTIFACT_ROOT / 'notebook' / EXPERIMENT_NAME / CORPUS_NAME / 'review.json'
from pilot_eval.workflow import _save_frozen
_save_frozen(CORPUS_REVIEW_PATH, CORPUS_REVIEW)
print('Reviewed corpus selection:', CORPUS_REVIEW_PATH)


## 8. Prepare or reuse the raw corpus (CPU)

Loads tokenizer files and streams the pinned FineWeb pool; no model weights or inference. Progress and errors are logged. A completed corpus is verified and reused without streaming or tokenizing again. An interrupted, unsealed preparation replays this bounded pool.


In [ ]:
runner.run([
    sys.executable, '-u', '-m', 'pilot_eval', 'traces-corpus-prepare',
    '--source-config', str(CORPUS_SOURCE), '--review', str(CORPUS_REVIEW_PATH),
    '--name', CORPUS_NAME, '--output-root', str(ARTIFACT_ROOT),
], env={'USE_TORCH': '0', 'USE_TF': '0'})
CORPUS_PATH = ARTIFACT_ROOT / 'plans' / CORPUS_NAME / 'traces.corpus.json'
session.select_corpus(CORPUS_PATH)
print('Frozen corpus:', CORPUS_PATH)


## 9. Verify saved corpus identities (CPU)

Checks corpus, tokenizer-file and input-parent hashes. Prints counts, tokenizer conventions and the new source indices. This verifies input preparation, not model capture or a scientific result. After a restart, run 1–2 and this cell to restore the selected corpus.


In [ ]:
state = session.restore()
assert state['corpus'].get('path'), 'Prepare and select the corpus in cell 8 first.'
CORPUS_PATH = Path(state['corpus']['path'])
runner.run([
    sys.executable, '-u', '-m', 'pilot_eval', 'traces-corpus-verify',
    '--config', str(CORPUS_PATH), '--output-root', str(ARTIFACT_ROOT),
])


## 10. Restore sources for the approved reference procedure (CPU)

Uses the completed measurement corpus and existing training records. No checkpoint audit, fine-tuning or answer generation is repeated. The separate background will contain exactly 2M raw tokens selected from a bounded 10,000-document scan.


In [ ]:
state = session.restore()
assert state['corpus']['state'].startswith('saved'), 'Complete corpus cells 7–9 first.'
CORPUS_PATH = Path(state['corpus']['path'])
for label in ['sft', 'grpo']:
    assert state['sources'][label]['state'].startswith('audited-config'), 'Audit the selected source first.'

sft_inputs = json.loads(Path(state['sources']['sft']['path']).read_text())
grpo_inputs = json.loads(Path(state['sources']['grpo']['path']).read_text())
SFT_TRAINING_CONFIG = ARTIFACT_ROOT / sft_inputs['source_config_path']
GRPO_TRAINING_CONFIG = ARTIFACT_ROOT / grpo_inputs['source_config_path']
REFERENCE_PATH = (ARTIFACT_ROOT / 'runs' / 'pilot-5' / EXPERIMENT_NAME
                  / 'analysis' / 'ticket03' / 'references' / 'freeze.json')
print('Measurement corpus:', CORPUS_PATH)
print('SFT training source:', SFT_TRAINING_CONFIG)
print('GRPO training source:', GRPO_TRAINING_CONFIG)
print('Background: 10,000 source documents scanned; 2,000,000 raw tokens selected.')
print('Primary: normalized-string overlap against the training-question reference.')


## 11. Prepare or resume task references (CPU)

Scans the pinned FineWeb source, excluding measurement documents and duplicates. Scan chunks are saved every 250 documents; an interrupted chunk can repeat. Then counts 512 questions, 512 gold solutions and all 4,096 accepted GRPO completions. Output: frozen references, counts and a top-30 review file. No lens or model is loaded. Initial real-data duration has not been measured.


In [ ]:
runner.run([
    sys.executable, '-u', '-m', 'pilot_eval', 'traces-reference-prepare',
    '--corpus', str(CORPUS_PATH),
    '--sft-config', str(SFT_TRAINING_CONFIG),
    '--grpo-config', str(GRPO_TRAINING_CONFIG),
    '--name', EXPERIMENT_NAME, '--output-root', str(ARTIFACT_ROOT),
], env={'USE_TORCH': '0', 'USE_TF': '0'})
print('Reference freeze:', REFERENCE_PATH)


## 12. Verify and inspect the reference preview (CPU)

Checks reference and source-input hashes. Inspect the three top-30 tables, including the short-reference flags. This is a vocabulary review, not a learned token list. Unexpected formatting, names or weak enrichment should be discussed before approval.


In [ ]:
from pilot_eval.trace_reference_workflow import verify_reference_run
from IPython.display import Markdown, display

REFERENCE_PATH = (ARTIFACT_ROOT / 'runs' / 'pilot-5' / EXPERIMENT_NAME
                  / 'analysis' / 'ticket03' / 'references' / 'freeze.json')
REFERENCE_SUMMARY = verify_reference_run(REFERENCE_PATH, ARTIFACT_ROOT)
print(json.dumps({key: value for key, value in REFERENCE_SUMMARY.items()
                  if key != 'procedure'}, indent=2))
display(Markdown((REFERENCE_PATH.parent / 'top30.md').read_text()))


## 13. Record the reference review (CPU)

Complete only after reading cell 12. Set the flag and add review notes. Exposure status describes whether Pilot 5 learned lens lists were seen before this freeze: `not-seen`, `seen-before-freeze`, or `unknown`. Unknown history is not recorded as confirmatory. This gate does not qualify a real lens or launch capture.


In [ ]:
REFERENCE_REVIEWED = False
REFERENCE_REVIEWER = 'Prasad'
REFERENCE_REVIEW_NOTES = ''
REFERENCE_EXPOSURE_STATUS = 'unknown'

assert REFERENCE_REVIEWED and REFERENCE_REVIEW_NOTES.strip(), 'Read the top-30 tables and record your review.'
runner.run([
    sys.executable, '-u', '-m', 'pilot_eval', 'traces-reference-review',
    '--config', str(REFERENCE_PATH), '--reviewer', REFERENCE_REVIEWER,
    '--notes', REFERENCE_REVIEW_NOTES,
    '--exposure-status', REFERENCE_EXPOSURE_STATUS,
    '--output-root', str(ARTIFACT_ROOT),
])


## 14. Prepare the four-document instrument check (CPU)

Selects the first two saved documents from each corpus group. Binds the exact inputs and both saved step-0 adapters. No model is loaded. The diagnostic is 4 documents × 3 conditions = 12 singleton forwards, with all 29 sites captured together. Source checks cover the sealed inputs and adapter files used here, without re-auditing every optimizer/checkpoint.

In [ ]:
state = session.restore()
assert state['corpus'].get('path'), 'Restore the existing corpus selection first.'
for arm in ['sft', 'grpo']:
    assert state['sources'][arm]['state'].startswith('audited-config'), 'Restore the audited source selection first.'
CAPTURE_DIAGNOSTIC_NAME = EXPERIMENT_NAME + '-capture-diagnostic-v1'
CAPTURE_DIAGNOSTIC_PATH = ARTIFACT_ROOT / 'plans' / CAPTURE_DIAGNOSTIC_NAME / 'traces.capture-diagnostic.json'
runner.run([
    sys.executable, '-u', '-m', 'pilot_eval', 'traces-capture-prepare',
    '--corpus', state['corpus']['path'],
    '--sft-config', state['sources']['sft']['path'],
    '--grpo-config', state['sources']['grpo']['path'],
    '--name', CAPTURE_DIAGNOSTIC_NAME, '--output-root', str(ARTIFACT_ROOT),
], env={'USE_TORCH': '0', 'USE_TF': '0'})
print('Saved diagnostic plan:', CAPTURE_DIAGNOSTIC_PATH)

## 15. Run or resume the bounded instrument diagnostic (GPU)

Fresh GPU session: run 1–2 and 4, then this cell after preparing 14. Loads one pinned FP32 model and both zero adapters. Captures all sites together, saves each completed condition and resumes only missing sealed work. Reports forwards and hash time separately. No trained checkpoint, generation, lens token list or batch-size search is run. Initial real duration is unmeasured; model download/setup can dominate. A changed runtime requires a separately named diagnostic, preserving old evidence.

Expected: `diagnostic_complete: true`, `qualified: false`, `state: awaiting-review`. Do not infer full GPU qualification or nonzero-delta precision from this zero check. Wait for child exit and CPU verification in 16 before releasing the GPU.

In [ ]:
CAPTURE_DIAGNOSTIC_PATH = (ARTIFACT_ROOT / 'plans'
    / (EXPERIMENT_NAME + '-capture-diagnostic-v1') / 'traces.capture-diagnostic.json')
assert CAPTURE_DIAGNOSTIC_PATH.is_file(), 'Prepare cell 14 first; it does not need a GPU.'
runner.run([
    sys.executable, '-u', '-m', 'pilot_eval', 'traces-capture-diagnose',
    '--config', str(CAPTURE_DIAGNOSTIC_PATH), '--output-root', str(ARTIFACT_ROOT),
], env={'USE_TF': '0'})

## 16. Verify saved diagnostic evidence (CPU)

Verifies saved inputs, condition seals, arrays, numerical evidence and base moments without loading a model. Can run immediately on the GPU runtime or after reconnecting on CPU with 1–2. Share the summary and any failure diagnostics for numerical-gate review. This step does not launch production capture.

In [ ]:
CAPTURE_DIAGNOSTIC_PATH = (ARTIFACT_ROOT / 'plans'
    / (EXPERIMENT_NAME + '-capture-diagnostic-v1') / 'traces.capture-diagnostic.json')
runner.run([
    sys.executable, '-u', '-m', 'pilot_eval', 'traces-capture-verify',
    '--config', str(CAPTURE_DIAGNOSTIC_PATH), '--output-root', str(ARTIFACT_ROOT),
], env={'USE_TORCH': '0', 'USE_TF': '0'})

## Next stages

Real diagnostic evidence still needs review. Community Jacobian-lens qualification and the reviewed production budget are separate tickets. No production capture, learned token lists, real null distribution or grader submission has run here.